# ReplayAxis · Two-view markerless motion analysis

Run the two code cells below for a dependency-free synthetic demonstration inside Jupyter. Orbit, zoom, play, pause, scrub or step through the reconstructed skeleton. The right panel shows source frame IDs, usable joints and reprojection error. The exported HTML opens directly in a browser.

### Capture requirements

The prototype reconstructs **one athlete in two fixed, overlapping views of the same take**. The clips must share a time reference. Metric reconstruction requires camera calibration that matches the videos' exact resolution, crop, zoom and placement. Moving cameras, unrelated takes, severe occlusion and poor camera separation can make depth unobservable. The output is a 3D skeleton rather than a photorealistic rendering.

In [ ]:
import json, math
from pathlib import Path
from html import escape
JOINTS = {0:'Head',11:'Left shoulder',12:'Right shoulder',13:'Left elbow',14:'Right elbow',
          15:'Left wrist',16:'Right wrist',23:'Left hip',24:'Right hip',
          25:'Left knee',26:'Right knee',27:'Left ankle',28:'Right ankle'}
BONES = [(0,11),(0,12),(11,12),(11,13),(13,15),(12,14),(14,16),
         (11,23),(12,24),(23,24),(23,25),(25,27),(24,26),(26,28)]
def dot(a,b):
    return sum(x*y for x,y in zip(a,b))
def diff(a,b):
    return [x-y for x,y in zip(a,b)]
def norm(a):
    return math.sqrt(dot(a,a))
def project(point,camera):
    q = [dot(row,point)+t for row,t in zip(camera['R'],camera['t'])]
    if q[2]<=0:
        return None
    u,v,w = [dot(row,q) for row in camera['K']]
    return [u/w,v/w]
def ray(pixel,camera):
    k=camera['K']
    q=[(pixel[0]-k[0][2])/k[0][0],(pixel[1]-k[1][2])/k[1][1],1]
    cols=list(zip(*camera['R']))
    center=[-dot(col,camera['t']) for col in cols]
    direction=[dot(col,q) for col in cols]
    length=norm(direction)
    return center,[x/length for x in direction]
def triangulate(a,b,camera_a,camera_b,max_error_px=10):
    ca,da=ray(a,camera_a)
    cb,db=ray(b,camera_b)
    c=max(-1.,min(1.,dot(da,db)))
    if math.degrees(math.acos(abs(c)))<2:
        return None,'weak geometry'
    d=diff(cb,ca)
    sa=(dot(d,da)-c*dot(d,db))/(1-c*c)
    sb=(c*dot(d,da)-dot(d,db))/(1-c*c)
    if min(sa,sb)<=0:
        return None,'behind camera'
    point=[(x+sa*u+y+sb*v)/2 for x,u,y,v in zip(ca,da,cb,db)]
    errors=[norm(diff(q,p)) for camera,p in ((camera_a,a),(camera_b,b))
            if (q:=project(point,camera)) is not None]
    if len(errors)!=2 or max(errors)>max_error_px:
        return None,'reprojection error'
    return point,sum(errors)/2
def reconstruct(a,b,cameras,confidence=.55,max_error_px=10):
    points,rejected,errors={},{},[]
    for j in JOINTS:
        if j not in a or j not in b or min(a[j][2],b[j][2])<confidence:
            rejected[j]='low visibility'
            continue
        point,quality=triangulate(a[j][:2],b[j][:2],*cameras,max_error_px)
        if point is None:
            rejected[j]=quality
        else:
            points[j]=point
            errors.append(quality)
    return dict(points=points,rejected=rejected,error_px=sum(errors)/len(errors) if errors else None)
def export_view(frames,title,stem,rate,units='m'):
    if not any(frame['points'] for frame in frames):
        raise ValueError('No landmarks reconstructed. Check synchronization, calibration, visibility, and camera separation.')
    payload=dict(frames=frames,title=title,bones=BONES,joints=JOINTS,rate=rate,units=units)
    data=json.dumps(payload,allow_nan=False)
    Path(stem+'.json').write_text(data,encoding='utf-8')
    html=VIEWER.replace('__DATA__',data.replace('<',r'\u003c'))
    Path(stem+'.html').write_text(html,encoding='utf-8')
    if 'get_ipython' in globals():
        from IPython.display import HTML,display
        display(HTML(f'<iframe title="ReplayAxis viewer" sandbox="allow-scripts" width="100%" height="620" style="border:0" srcdoc="{escape(html,quote=True)}"></iframe>'))
    print(f'Saved {stem}.html and {stem}.json')

VIEWER = '<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>ReplayAxis · Motion Lab</title>\n<style>\n*{box-sizing:border-box}body{margin:0;background:#10171c;color:#ebf3f1;font:15px system-ui}main{max-width:1140px;margin:auto;padding:22px}header{border-top:3px solid #82e6be;padding:14px 0 16px}h1,h2,p{margin:0}h1{font-size:26px}h2{font-size:16px;margin-bottom:14px}p,small{color:#afc2c5}header p{margin-top:5px}.grid{display:grid;grid-template-columns:minmax(0,1fr) 240px;gap:12px}canvas,aside{background:#182329;border:1px solid #354750;border-radius:8px}canvas{display:block;width:100%;height:358px;touch-action:none;cursor:grab}aside{padding:16px}dl{display:grid;gap:8px;margin:0}dt{color:#afc2c5}dd{margin:0 0 7px;font:16px ui-monospace,monospace}.bar{display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin:12px 0}button{background:#23343d;color:#eef8f4;border:1px solid #45616a;border-radius:6px;padding:8px 12px;cursor:pointer}button:hover,button:focus-visible{border-color:#82e6be}input{flex:1;min-width:110px;accent-color:#82e6be}@media(max-width:650px){.grid{grid-template-columns:1fr}canvas{height:290px}aside{padding:12px}dl{grid-template-columns:auto 1fr}}\n</style>\n<main><header><h1>ReplayAxis</h1><p id="title"></p></header>\n<div class="grid"><canvas tabindex="0" aria-label="3D skeleton. Drag or use arrow keys to orbit."></canvas><aside><h2>Frame diagnostics</h2><dl><dt>Time</dt><dd id="time"></dd><dt>Source frames · A / B</dt><dd id="source"></dd><dt>Landmarks reconstructed</dt><dd id="count"></dd><dt>Mean reprojection error</dt><dd id="err"></dd></dl></aside></div>\n<div class="bar"><button id="play">Play</button><button id="prev" aria-label="Previous frame">−1 frame</button><button id="next" aria-label="Next frame">+1 frame</button><input type="range" id="scrub" aria-label="Playback frame"><button id="reset">Reset view</button></div>\n<small>Drag or use arrow keys to orbit. Scroll over the skeleton to zoom. Hidden joints are omitted.</small>\n</main>\n<script>\nconst data=__DATA__;\nconst el=id=>document.getElementById(id);\nconst canvas=document.querySelector(\'canvas\'),ctx=canvas.getContext(\'2d\');\nconst frames=data.frames;\nel(\'title\').textContent=data.title;\nconst points=frames.flatMap(f=>Object.values(f.points));\nconst bounds=[0,1,2].map(axis=>points.reduce(([lo,hi],p)=>[Math.min(lo,p[axis]),Math.max(hi,p[axis])],[Infinity,-Infinity]));\nconst center=bounds.map(([lo,hi])=>(lo+hi)/2);\nconst extent=Math.max(.001,...bounds.map(([lo,hi])=>hi-lo));\nlet i=0,yaw=-.44,pitch=.12,zoom=1,drag=null,timer=null;\nconst scrub=el(\'scrub\');scrub.min=0;scrub.max=frames.length-1;scrub.value=0;\nfunction render(){\n const w=canvas.clientWidth,h=canvas.clientHeight,dpr=devicePixelRatio||1;\n if(canvas.width!==Math.round(w*dpr)||canvas.height!==Math.round(h*dpr)){canvas.width=Math.round(w*dpr);canvas.height=Math.round(h*dpr)}\n ctx.setTransform(dpr,0,0,dpr,0,0);ctx.clearRect(0,0,w,h);\n ctx.strokeStyle=\'#23353e\';ctx.lineWidth=1;\n for(let x=0;x<w;x+=24){ctx.beginPath();ctx.moveTo(x,0);ctx.lineTo(x,h);ctx.stroke()}\n for(let y=0;y<h;y+=24){ctx.beginPath();ctx.moveTo(0,y);ctx.lineTo(w,y);ctx.stroke()}\n const scale=Math.min(w,h)*.76*zoom/extent;\n function screen(p){\n  const [x,y,z]=p.map((v,j)=>v-center[j]);\n  const a=x*Math.cos(yaw)+z*Math.sin(yaw),b=z*Math.cos(yaw)-x*Math.sin(yaw);\n  return [w/2+a*scale,h/2+(y*Math.cos(pitch)-b*Math.sin(pitch))*scale];\n }\n const f=frames[i], projected=Object.fromEntries(Object.entries(f.points).map(([j,p])=>[j,screen(p)]));\n ctx.strokeStyle=\'#82e6be\';ctx.lineCap=\'round\';ctx.lineWidth=3;\n for(const [a,b] of data.bones){\n  if(!projected[a]||!projected[b])continue;\n  ctx.beginPath();ctx.moveTo(...projected[a]);ctx.lineTo(...projected[b]);ctx.stroke();\n }\n ctx.fillStyle=\'#effff8\';\n for(const [x,y] of Object.values(projected)){ctx.beginPath();ctx.arc(x,y,4,0,2*Math.PI);ctx.fill()}\n el(\'time\').textContent=f.time.toFixed(3)+\' s\';\n el(\'source\').textContent=f.source_frames.join(\' / \');\n el(\'count\').textContent=Object.keys(projected).length+\' / \'+Object.keys(data.joints).length;\n el(\'err\').textContent=f.error_px===null?\'—\':f.error_px.toFixed(2)+\' px\';\n scrub.value=i;\n}\nfunction show(index){i=Math.max(0,Math.min(frames.length-1,index));render()}\nfunction stop(){clearInterval(timer);timer=null;el(\'play\').textContent=\'Play\'}\nel(\'play\').onclick=()=>{if(timer!==null){stop();return}if(i===frames.length-1)show(0);el(\'play\').textContent=\'Pause\';timer=setInterval(()=>{if(i===frames.length-1)stop();else show(i+1)},1000/data.rate)};\nel(\'prev\').onclick=()=>{stop();show(i-1)};el(\'next\').onclick=()=>{stop();show(i+1)};\nscrub.oninput=()=>{stop();show(Number(scrub.value))};\nel(\'reset\').onclick=()=>{yaw=-.44;pitch=.12;zoom=1;render()};\ncanvas.onpointerdown=e=>{drag=[e.clientX,e.clientY];canvas.setPointerCapture(e.pointerId)};\ncanvas.onpointermove=e=>{if(!drag)return;yaw+=(e.clientX-drag[0])*.012;pitch+=(e.clientY-drag[1])*.012;drag=[e.clientX,e.clientY];render()};\ncanvas.onlostpointercapture=()=>drag=null;\ncanvas.addEventListener(\'wheel\',e=>{e.preventDefault();zoom=Math.min(5,Math.max(.25,zoom*Math.exp(-e.deltaY*.001)));render()},{passive:false});\ncanvas.onkeydown=e=>{const key={ArrowLeft:[-.1,0],ArrowRight:[.1,0],ArrowUp:[0,-.1],ArrowDown:[0,.1]}[e.key];if(key){e.preventDefault();yaw+=key[0];pitch+=key[1];render()}};\nnew ResizeObserver(render).observe(canvas);render();\n</script></html>\n'

In [ ]:
identity=[[1,0,0],[0,1,0],[0,0,1]]
k=[[950,0,640],[0,950,360],[0,0,1]]
a,b=[dict(K=k,R=identity,t=t) for t in ([0,0,0],[-1.15,0,0])]
base={0:(0,-.9,0),11:(-.2,-.62,0),12:(.2,-.62,0),13:(-.35,-.36,0),14:(.35,-.36,0),
      15:(-.4,-.18,0),16:(.4,-.18,0),23:(-.13,0,0),24:(.13,0,0),
      25:(-.15,.42,.04),26:(.15,.42,.04),27:(-.16,.81,0),28:(.16,.81,0)}
frames=[]
worst=0
for i in range(120):
    phase=2*math.pi*i/45
    truth={j:[x+.008*i,y,4.6+z+(.22*math.sin(phase+math.pi*(j%2)) if j in (13,14,15,16,25,26,27,28) else 0)]
           for j,(x,y,z) in base.items()}
    first={j:[*project(p,a),1.] for j,p in truth.items()}
    second={j:[*project(p,b),1.] for j,p in truth.items()}
    if 40<=i<57:
        second[15][2]=0.
    f=reconstruct(first,second,(a,b))
    assert len(f['points'])==(12 if 40<=i<57 else 13)
    worst=max(worst,max(norm(diff(p,truth[j])) for j,p in f['points'].items()))
    f.update(time=i/30,source_frames=[i,i])
    frames.append(f)
assert worst<1e-8,worst
assert triangulate([640,360],[640,360],a,b)[1]=='weak geometry'
assert triangulate([640,360],[400,480],a,b)[1]=='reprojection error'
print(f'Synthetic check passed: {len(frames)} frames, max error {worst:.2e} m, intentional occlusion withheld.')
export_view(frames,'Synthetic motion · known calibration','ReplayAxis_Demo',30)

## Analyze your footage

Use a Python 3.11 or 3.12 Jupyter kernel for the optional real-video cells. Install the packages below once, then restart the kernel and run the core cell again. OpenCV decodes videos and corrects lens distortion. MediaPipe estimates image landmarks and ReplayAxis triangulates matched landmarks. The model downloads on first use. You can also place Google's pose_landmarker_lite.task file beside the notebook.

Use constant-frame-rate clips. Set matching start times from a common event, such as a visible clap or flash, or use synchronized camera timestamps. This pipeline selects the nearest frame in each view. Fast motion requires tight synchronization, and start offsets alone do not correct clock drift. Source times are included in the JSON for auditing.

In [ ]:
# Uncomment to install in the current Jupyter kernel:
# %pip install "mediapipe>=0.10,<0.11" "opencv-contrib-python>=4.8,<5" "numpy>=1.24,<3"

In [ ]:
import contextlib
import urllib.request
import cv2
import mediapipe as mp
import numpy as np
MODEL=Path('pose_landmarker_lite.task')
MODEL_URL=('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
           'pose_landmarker_lite/float16/1/pose_landmarker_lite.task')

def detect_pose(detector, image, frame_index, fps, camera):
    rgb=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    height,width=rgb.shape[:2]
    result=detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb),
                                     round(frame_index*1000/fps))
    if not result.pose_landmarks:
        return {}
    pose=result.pose_landmarks[0]
    pixels=np.array([[[pose[j].x*width,pose[j].y*height]] for j in JOINTS],dtype=float)
    k=np.asarray(camera['K'],dtype=float)
    lens=np.asarray(camera.get('dist',[0,0,0,0,0]),dtype=float)
    corrected=cv2.undistortPoints(pixels,k,lens,P=k).reshape(-1,2)
    return {j:[float(x),float(y),min(pose[j].visibility,pose[j].presence)]
            for j,(x,y) in zip(JOINTS,corrected) if math.isfinite(x) and math.isfinite(y)
            and 0<=pixels[list(JOINTS).index(j),0,0]<width and 0<=pixels[list(JOINTS).index(j),0,1]<height}

def analyze(video_a,video_b,calibration,start_a=0.,start_b=0.,duration=5.,rate=20.):
    if not all(math.isfinite(v) for v in (start_a,start_b,duration,rate)) or min(start_a,start_b)<0 or min(duration,rate)<=0:
        raise ValueError('Start times must be nonnegative and duration and rate positive.')
    cameras=json.loads(Path(calibration).read_text())['cameras']
    if len(cameras)!=2:
        raise ValueError('Calibration must contain exactly two cameras in video order.')
    for camera in cameras:
        k,r,t=(np.asarray(camera[key],dtype=float) for key in ('K','R','t'))
        if (k.shape!=(3,3) or r.shape!=(3,3) or t.shape!=(3,) or
            not all(np.isfinite(item).all() for item in (k,r,t)) or
            min(k[0,0],k[1,1])<=0 or abs(k[0,1])>1e-8 or
            not np.allclose(k[2],[0,0,1]) or not np.allclose(r@r.T,np.eye(3),atol=1e-4)
            or not np.isclose(np.linalg.det(r),1,atol=1e-4)):
            raise ValueError('Invalid calibration: expected zero-skew K, proper rotation R and translation t.')
    centers=[ray([0,0],camera)[0] for camera in cameras]
    if norm(diff(*centers))<1e-6:
        raise ValueError('Camera centers must be distinct.')
    if not MODEL.exists():
        partial=MODEL.with_suffix('.download')
        try:
            urllib.request.urlretrieve(MODEL_URL,partial)
            partial.replace(MODEL)
        finally:
            partial.unlink(missing_ok=True)
    options=mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,num_poses=1)
    frames=[]
    with contextlib.ExitStack() as stack:
        captures=[]
        for path,camera in zip((video_a,video_b),cameras):
            capture=cv2.VideoCapture(str(path))
            stack.callback(capture.release)
            if not capture.isOpened():
                raise ValueError(f'Cannot open video: {path}')
            size=[int(capture.get(prop)) for prop in (cv2.CAP_PROP_FRAME_WIDTH,cv2.CAP_PROP_FRAME_HEIGHT)]
            if size!=camera['image_size']:
                raise ValueError(f'{path}: size {size} does not match calibration {camera["image_size"]}.')
            captures.append(capture)
        fps=[c.get(cv2.CAP_PROP_FPS) for c in captures]
        if not all(math.isfinite(f) and 0<f<1000 for f in fps):
            raise ValueError('Use readable constant-frame-rate sources with valid FPS metadata.')
        rate=min(rate,*fps)
        detectors=[stack.enter_context(mp.tasks.vision.PoseLandmarker.create_from_options(options)) for _ in captures]
        last=[-1,-1]
        for tick in range(math.ceil(duration*rate)):
            elapsed=tick/rate
            indexes=[round((start+elapsed)*f) for start,f in zip((start_a,start_b),fps)]
            views=[]
            for slot,(cap,index) in enumerate(zip(captures,indexes)):
                while last[slot]<index-1:
                    if not cap.grab():
                        break
                    last[slot]+=1
                ok,img=cap.read()
                if not ok:
                    break
                last[slot]=index
                views.append(detect_pose(detectors[slot],img,index,fps[slot],cameras[slot]))
            if len(views)!=2:
                break
            frame=reconstruct(*views,cameras)
            frame.update(time=elapsed,source_frames=indexes,
                         source_times=[index/f for index,f in zip(indexes,fps)])
            frames.append(frame)
    export_view(frames,'Two-view reconstruction · calibrated','ReplayAxis_Real_Clip',rate)
    print(f'Reconstructed {sum(bool(f["points"]) for f in frames)} of {len(frames)} paired frames.')
    return frames

### Optional checkerboard calibration

If you already have a calibration, skip this section. Otherwise capture at least ten paired stills of a checkerboard, in varied positions and orientations throughout the capture volume. Keep the camera settings and placement identical to the videos, and keep the board still during each pair. Put the files in calibration_A and calibration_B with matching filenames. Specify internal corners (columns, rows) and measured square size in metres. Use an asymmetric placement or mark the board outside the corner grid so corner ordering can be checked in both views.

The JSON requires an ordered cameras list. Each entry contains K (3×3 intrinsics, zero skew), R (3×3 world-to-camera rotation), t (three world-to-camera translation values in metres), image_size ([width, height]) and optional dist (OpenCV distortion coefficients). Both transforms must use the same world coordinate system. Verify the calibration with a known distance that was not used in the fit.

In [ ]:
def calibrate_checkerboard(folder_a,folder_b,inner_corners=(9,6),square_m=.025,
                           output='stereo_calibration.json'):
    if min(inner_corners)<3 or square_m<=0:
        raise ValueError('Specify at least 3 × 3 inner corners and a positive square size in metres.')
    board=np.zeros((inner_corners[0]*inner_corners[1],3),np.float32)
    board[:,:2]=np.mgrid[0:inner_corners[0],0:inner_corners[1]].T.reshape(-1,2)*square_m
    points,first,second=[],[],[]
    image_size=None
    for path in sorted(Path(folder_a).iterdir()):
        other=Path(folder_b)/path.name
        if not path.is_file() or not other.is_file():
            continue
        a,b=(cv2.imread(str(p),cv2.IMREAD_GRAYSCALE) for p in (path,other))
        if a is None or b is None:
            continue
        size=a.shape[::-1]
        if a.shape!=b.shape or (image_size is not None and size!=image_size):
            raise ValueError('This calibration helper requires equal image dimensions in both cameras and all pairs.')
        ok_a,ca=cv2.findChessboardCornersSB(a,inner_corners)
        ok_b,cb=cv2.findChessboardCornersSB(b,inner_corners)
        if ok_a and ok_b:
            image_size=size
            points.append(board.copy())
            first.append(ca)
            second.append(cb)
    if len(points)<10:
        raise ValueError(f'Only {len(points)} valid pairs. Provide at least 10 varied checkerboard poses.')
    rms_a,ka,da,*_=cv2.calibrateCamera(points,first,image_size,None,None)
    rms_b,kb,db,*_=cv2.calibrateCamera(points,second,image_size,None,None)
    rms,ka,da,kb,db,r,t,*_=cv2.stereoCalibrate(points,first,second,ka,da,kb,db,
                                                image_size,flags=cv2.CALIB_FIX_INTRINSIC)
    cameras=[dict(K=k.tolist(),dist=d.ravel().tolist(),R=rotation.tolist(),
                  t=translation.ravel().tolist(),image_size=list(image_size))
             for k,d,rotation,translation in [(ka,da,np.eye(3),np.zeros(3)),(kb,db,r,t)]]
    data=dict(cameras=cameras,units='m',pairs=len(points),stereo_rms_px=float(rms))
    Path(output).write_text(json.dumps(data,indent=2))
    print(f'{len(points)} pairs · camera RMS {rms_a:.2f}, {rms_b:.2f} px · stereo RMS {rms:.2f} px')
    if rms>1:
        print('Review image pairs and corner ordering before analysis. Independent geometry validation is still needed.')
    return data

# calibrate_checkerboard('calibration_A','calibration_B',inner_corners=(9,6),square_m=.025)

### Run the analysis

Set the paths and matching times below. For example, if a clap occurs at 0.60 s in angle A and 1.05 s in angle B, enter those two values. Start with a short segment. Keep a single athlete in both views and check that landmarks belong to the same person. The notebook withholds low-visibility joints, weak ray intersections and high-reprojection-error estimates. Reprojection error helps diagnose geometry and does not measure real-world anatomical accuracy.

In [ ]:
VIDEO_A = Path('angle_A.mp4')
VIDEO_B = Path('angle_B.mp4')
CALIBRATION = Path('stereo_calibration.json')
START_A_SECONDS = 0.0
START_B_SECONDS = 0.0
DURATION_SECONDS = 5.0
SAMPLE_HZ = 20.0

# frames = analyze(VIDEO_A, VIDEO_B, CALIBRATION, START_A_SECONDS, START_B_SECONDS,
#                  DURATION_SECONDS, SAMPLE_HZ)